# g4_stl_first_hunyuan3d_shape_debug_s40_n1 Colab Launcher

Run the code cell below in Colab to download, verify, and launch the packaged benchmark.


In [ ]:
# Paste this into one Colab Python cell to download and launch the packaged benchmark.
import hashlib
import os
import pathlib
import subprocess
import tarfile
import urllib.request

PAYLOAD_URL = "https://raw.githubusercontent.com/DrStrangel0ve/3dprintpic/c0d5b676aac21c98f9e5c5d3461b555768df9d5e/colab_payloads/modelnet10_heldout1_stl_first_hunyuan3d_shape_debug_colab_inputs.tar.gz"
ARCHIVE_PATH = pathlib.Path("/content/modelnet10_heldout1_stl_first_hunyuan3d_shape_debug_colab_inputs.tar.gz")
EXTRACT_ROOT = pathlib.Path("/content/3dprintpic_colab_inputs/modelnet10_heldout1_stl_first_hunyuan3d_shape_debug")
EXPECTED_SHA256 = "e0e7ac630a64268d3a9c153be91ed349570448b6d7b03ed8027c45ca7a1f1269"
EXPECTED_SIZE = 133283

ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'Downloading {PAYLOAD_URL}')
with urllib.request.urlopen(PAYLOAD_URL) as response:
    payload = response.read()
ARCHIVE_PATH.write_bytes(payload)

actual_size = ARCHIVE_PATH.stat().st_size
actual_sha256 = hashlib.sha256(payload).hexdigest()
print({'archive': str(ARCHIVE_PATH), 'bytes': actual_size, 'sha256': actual_sha256})
if actual_size != EXPECTED_SIZE:
    raise SystemExit(f'archive size mismatch: expected {EXPECTED_SIZE} got {actual_size}')
if actual_sha256 != EXPECTED_SHA256:
    raise SystemExit(f'archive sha256 mismatch: expected {EXPECTED_SHA256} got {actual_sha256}')

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
run_script = EXTRACT_ROOT / 'run_colab_eval.sh'
with tarfile.open(ARCHIVE_PATH, 'r:gz') as tar:
    try:
        member = tar.getmember('run_colab_eval.sh')
    except KeyError as exc:
        raise SystemExit('archive does not contain run_colab_eval.sh; regenerate with --include-run-script') from exc
    extracted = tar.extractfile(member)
    if extracted is None:
        raise SystemExit('could not read run_colab_eval.sh from archive')
    run_script.write_bytes(extracted.read())
run_script.chmod(0o755)

env = os.environ.copy()
env['EXTRACT_ROOT'] = str(EXTRACT_ROOT)
env['EXPECTED_SHA256'] = EXPECTED_SHA256
print(f'Launching {run_script} with {ARCHIVE_PATH}')
subprocess.run(['bash', str(run_script), str(ARCHIVE_PATH)], check=True, env=env)
